# 财经大数据收集与预处理 - 教改项目演示

本 Notebook 演示完整的数据采集与预处理流程：
1. 多源数据采集（天池、Kaggle、爬虫）
2. 数据清洗与预处理流水线

In [ ]:
import sys
sys.path.insert(0, '..')

from src.collectors import TianchiCollector, KaggleCollector, FinanceCrawler
from src.preprocessors import PreprocessingPipeline, MissingValueHandler, OutlierDetector, DataNormalizer, FeatureEngineer
from src.utils import auto_load, setup_logger
import pandas as pd
import numpy as np

## 1. 多源数据采集

In [ ]:
# 查看天池可用数据集
tc = TianchiCollector()
tc.list_datasets()

In [ ]:
# 查看 Kaggle 可用数据集
kc = KaggleCollector()
kc.list_datasets()

In [ ]:
# 爬虫采集财经资讯（示例）
crawler = FinanceCrawler(request_delay=2.0)
# results = crawler.crawl_and_save(sources=['eastmoney'], max_pages=1, format='json')

## 2. 生成示例金融数据

In [ ]:
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'loan_id': range(1, n + 1),
    'age': np.random.randint(22, 65, n),
    'income': np.random.lognormal(10.5, 0.8, n).round(2),
    'loan_amount': np.random.lognormal(11, 0.6, n).round(2),
    'credit_score': np.clip(np.random.normal(680, 80, n), 300, 850).astype(int),
    'debt_ratio': np.clip(np.random.beta(2, 5, n), 0, 1).round(4),
    'employment_years': np.random.exponential(5, n).round(1),
    'default': np.random.binomial(1, 0.15, n),
})

# 注入缺失值
for col in ['income', 'credit_score', 'debt_ratio']:
    mask = np.random.random(n) < 0.05
    df.loc[mask, col] = np.nan

# 注入异常值
df.loc[np.random.choice(n, 5), 'income'] = df['income'].mean() * 50
df.loc[np.random.choice(n, 3), 'loan_amount'] = -9999

print(f'数据形状: {df.shape}')
df.head(10)

## 3. 缺失值分析与处理

In [ ]:
handler = MissingValueHandler()
report = handler.analyze_missing(df)

In [ ]:
# 自动填充
df_filled = handler.auto_fill(df)
print(f'填充后缺失值: {df_filled.isnull().sum().sum()}')

## 4. 异常值检测

In [ ]:
detector = OutlierDetector()

# IQR 法检测
results = detector.detect_iqr(df_filled, columns=['income', 'loan_amount', 'credit_score'])
print()
detector.get_outlier_report()

In [ ]:
# 截断异常值
df_clipped = detector.clip_outliers(df_filled, method='iqr')
print('异常值截断完成')

## 5. 数据标准化

In [ ]:
normalizer = DataNormalizer()

# Robust 标准化（对异常值鲁棒）
num_cols = df_clipped.select_dtypes(include=[np.number]).columns.tolist()
df_scaled = normalizer.robust_scale(df_clipped, columns=num_cols)
df_scaled.describe()

## 6. 一键流水线处理

In [ ]:
# 使用流水线一键处理
pipeline = PreprocessingPipeline()

# 重新生成原始数据
np.random.seed(42)
n = 1000
df_raw = pd.DataFrame({
    'loan_id': range(1, n + 1),
    'age': np.random.randint(22, 65, n),
    'income': np.random.lognormal(10.5, 0.8, n).round(2),
    'loan_amount': np.random.lognormal(11, 0.6, n).round(2),
    'credit_score': np.clip(np.random.normal(680, 80, n), 300, 850).astype(int),
    'debt_ratio': np.clip(np.random.beta(2, 5, n), 0, 1).round(4),
    'employment_years': np.random.exponential(5, n).round(1),
    'default': np.random.binomial(1, 0.15, n),
})
for col in ['income', 'credit_score', 'debt_ratio']:
    mask = np.random.random(n) < 0.05
    df_raw.loc[mask, col] = np.nan
df_raw.loc[np.random.choice(n, 5), 'income'] = df_raw['income'].mean() * 50
df_raw.loc[np.random.choice(n, 3), 'loan_amount'] = -9999

df_result = pipeline.run(df_raw)
print(pipeline.get_pipeline_report())

In [ ]:
# 保存结果
df_result.to_csv('../data/processed/notebook_result.csv', index=False, encoding='utf-8-sig')
pipeline.save_config()
print('处理完成！')